# Taller ANN Multiclass — Clasificación de Credit Score
> **Objetivo:** Clasificar clientes en 3 categorías de riesgo crediticio (0=Bueno, 1=Estándar, 2=Deficiente)  
> **Modelo:** Red Neuronal Artificial (ANN) Multiclase con Keras/TensorFlow

## Paso 0 — Instalación de dependencias
> Ejecuta esta celda **una sola vez**

In [1]:
!pip install pandas numpy scikit-learn tensorflow matplotlib seaborn

## Paso 1 — Importación de librerías

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

# Reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

print('✅ Librerías importadas correctamente')
print(f'   TensorFlow: {tf.__version__}')
print(f'   Pandas    : {pd.__version__}')

✅ Librerías importadas correctamente
   TensorFlow: 2.19.0
   Pandas    : 2.2.2


## Paso 2 — Carga del dataset

In [16]:
import os

# Verificar directorio de trabajo
print(f'📁 Directorio actual: {os.getcwd()}')
print(f'📄 Archivos disponibles: {os.listdir()}')

📁 Directorio actual: /content
📄 Archivos disponibles: ['.config', 'sample_data']


In [19]:
df = pd.read_csv('/content/riesgo.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/riesgo.csv'

In [ ]:
# Cargar el dataset CSV
df = pd.read_csv('/content/riesgo.csv')

print(f'✅ Dataset cargado correctamente')
print(f'📐 Filas    : {df.shape[0]}')
print(f'📐 Columnas : {df.shape[1]}')

KeyboardInterrupt: 

##  Paso 3 — Exploración del dataset (EDA)

In [ ]:
print('📋 Primeras 5 filas:')
display(df.head())

In [ ]:
print('🔍 Tipos de datos por columna:')
display(df.dtypes.to_frame(name='Tipo'))

In [ ]:
print('❓ Valores nulos por columna:')
nulos = df.isnull().sum()
display(nulos[nulos > 0].to_frame(name='Nulos'))

In [ ]:
print('📊 Estadísticas descriptivas:')
display(df.describe())

In [ ]:
# Distribución de la variable objetivo
print('📈 Distribución de Credit_Score:')
print(df['Credit_Score'].value_counts())

conteo    = df['Credit_Score'].value_counts().sort_index()
etiquetas = {0: 'Bueno (0)', 1: 'Estándar (1)', 2: 'Deficiente (2)'}

plt.figure(figsize=(6, 4))
ax = conteo.plot(kind='bar',
                 color=['#2ecc71', '#f39c12', '#e74c3c'],
                 edgecolor='black')
ax.set_title('Distribución de Credit Score', fontsize=14, fontweight='bold')
ax.set_xlabel('Categoría de riesgo')
ax.set_ylabel('Número de clientes')
ax.set_xticklabels([etiquetas[i] for i in conteo.index], rotation=0)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

## Paso 4 — Preprocesamiento

In [ ]:
# Eliminar columnas irrelevantes
cols_drop = ['Customer_ID', 'Name', 'SSN', 'Type_of_Loan']
df.drop(columns=cols_drop, inplace=True)
print(f'🗑️  Columnas eliminadas: {cols_drop}')

# Separar features y target
X = df.drop(columns=['Credit_Score'])
y = df['Credit_Score']

print(f'✅ Features (X): {X.shape}')
print(f'✅ Target  (y): {y.shape}')

In [ ]:
# Identificar columnas numéricas y categóricas
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

print(f'🔤 Variables categóricas ({len(cat_cols)}): {cat_cols}')
print(f'🔢 Variables numéricas   ({len(num_cols)}): {num_cols}')

In [ ]:
# Imputar valores nulos
# Numéricas → mediana | Categóricas → moda
for col in num_cols:
    X[col] = X[col].fillna(X[col].median())

for col in cat_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

print(f'✅ Valores nulos imputados')
print(f'   Nulos restantes: {X.isnull().sum().sum()}')

In [ ]:
# Codificación de variables categóricas (Label Encoding)
le = LabelEncoder()
for col in cat_cols:
    X[col] = le.fit_transform(X[col].astype(str))
    print(f'   🏷️  {col} → clases: {list(le.classes_)}')

print('\n✅ Codificación completada')

In [ ]:
# Normalización con StandardScaler
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print('✅ Normalización aplicada (StandardScaler)')
print(f'   Media  ≈ {X_scaled.mean().mean():.4f}')
print(f'   Desv   ≈ {X_scaled.std().mean():.4f}')

## Paso 5 — Reducción de dimensionalidad (PCA)

In [ ]:
# Calcular varianza explicada acumulada
pca_full           = PCA()
pca_full.fit(X_scaled)
varianza_acumulada = np.cumsum(pca_full.explained_variance_ratio_)

# Componentes necesarios para ≥ 90% de varianza
n_components = np.argmax(varianza_acumulada >= 0.90) + 1
print(f'📐 Componentes para ≥ 90% de varianza: {n_components}')

# Gráfico varianza acumulada
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(varianza_acumulada) + 1), varianza_acumulada,
         marker='o', markersize=5, color='#3498db', linewidth=2)
plt.axhline(y=0.90, color='red',   linestyle='--', label='90% varianza')
plt.axvline(x=n_components, color='green', linestyle='--',
            label=f'n={n_components} componentes')
plt.xlabel('Número de componentes principales')
plt.ylabel('Varianza explicada acumulada')
plt.title('PCA – Varianza Explicada Acumulada', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Aplicar PCA con n_components óptimo
pca   = PCA(n_components=n_components, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'✅ PCA aplicado')
print(f'   Features originales : {X_scaled.shape[1]}')
print(f'   Componentes finales : {X_pca.shape[1]}')
print(f'   Varianza explicada  : {pca.explained_variance_ratio_.sum():.4f}')

## Paso 6 — División Train / Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'📦 Entrenamiento : {X_train.shape[0]} muestras ({X_train.shape[0]/len(y)*100:.0f}%)')
print(f'📦 Prueba        : {X_test.shape[0]}  muestras ({X_test.shape[0]/len(y)*100:.0f}%)')

# One-Hot Encoding del target (requerido por softmax)
NUM_CLASES   = 3
y_train_cat  = to_categorical(y_train, num_classes=NUM_CLASES)
y_test_cat   = to_categorical(y_test,  num_classes=NUM_CLASES)

print(f'\n✅ One-Hot Encoding aplicado')
print(f'   y_train shape: {y_train_cat.shape}')
print(f'   y_test  shape: {y_test_cat.shape}')

## Paso 7 — Construcción del modelo ANN

In [ ]:
input_dim = X_train.shape[1]

modelo = Sequential([
    # Capa de entrada
    Dense(128, activation='relu', input_shape=(input_dim,), name='capa_entrada'),
    BatchNormalization(),
    Dropout(0.3),

    # Capa oculta 1
    Dense(64, activation='relu', name='capa_oculta_1'),
    BatchNormalization(),
    Dropout(0.3),

    # Capa oculta 2
    Dense(32, activation='relu', name='capa_oculta_2'),
    Dropout(0.2),

    # Capa de salida — 3 neuronas (una por clase), activación softmax
    Dense(NUM_CLASES, activation='softmax', name='capa_salida')
])

modelo.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

modelo.summary()

## Paso 8 — Entrenamiento

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

historia = modelo.fit(
    X_train, y_train_cat,
    epochs=100,
    batch_size=64,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=1
)

print(f'\n✅ Entrenamiento finalizado en {len(historia.history["loss"])} épocas')

## Paso 9 — Curvas de aprendizaje

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pérdida
axes[0].plot(historia.history['loss'],     label='Entrenamiento', color='#3498db', lw=2)
axes[0].plot(historia.history['val_loss'], label='Validación',    color='#e74c3c', lw=2, ls='--')
axes[0].set_title('Pérdida (Loss)',         fontsize=13, fontweight='bold')
axes[0].set_xlabel('Épocas')
axes[0].set_ylabel('Categorical Crossentropy')
axes[0].legend()
axes[0].grid(alpha=0.4)

# Exactitud
axes[1].plot(historia.history['accuracy'],     label='Entrenamiento', color='#2ecc71', lw=2)
axes[1].plot(historia.history['val_accuracy'], label='Validación',    color='#f39c12', lw=2, ls='--')
axes[1].set_title('Exactitud (Accuracy)',       fontsize=13, fontweight='bold')
axes[1].set_xlabel('Épocas')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.4)

plt.suptitle('Curvas de Aprendizaje – ANN Multiclase',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Paso 10 — Evaluación del modelo

In [ ]:
loss_test, acc_test = modelo.evaluate(X_test, y_test_cat, verbose=0)

print(f'🎯 Loss     (Test): {loss_test:.4f}')
print(f'🎯 Accuracy (Test): {acc_test:.4f}  →  {acc_test*100:.2f}%')

In [ ]:
# Reporte de clasificación
y_pred = np.argmax(modelo.predict(X_test), axis=1)

print('📋 Reporte de Clasificación:')
print(classification_report(
    y_test, y_pred,
    target_names=['Bueno (0)', 'Estándar (1)', 'Deficiente (2)']
))

In [ ]:
# Matriz de confusión
cm   = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Bueno (0)', 'Estándar (1)', 'Deficiente (2)']
).plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Matriz de Confusión – ANN Multiclase', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Paso 11 — Predicción sobre nuevos clientes

In [ ]:
etiquetas  = {0: 'Bueno', 1: 'Estándar', 2: 'Deficiente'}
n_ejemplos = 5
X_ejemplo  = X_test[:n_ejemplos]
y_real     = y_test.values[:n_ejemplos]
y_pred_ej  = np.argmax(modelo.predict(X_ejemplo), axis=1)

print(f'{"Cliente":>8} | {"Real":>12} | {"Predicción":>12} | {"¿Correcto?":>10}')
print('-' * 52)
for i in range(n_ejemplos):
    correcto = '✅' if y_real[i] == y_pred_ej[i] else '❌'
    print(f'{i+1:>8} | {etiquetas[y_real[i]]:>12} | {etiquetas[y_pred_ej[i]]:>12} | {correcto:>10}')

## Paso 12 — Guardar el modelo

In [ ]:
modelo.save('modelo_ann_credit_score.keras')
print('✅ Modelo guardado como: modelo_ann_credit_score.keras')
print('\n   Para cargarlo en el futuro:')
print('   modelo = tf.keras.models.load_model("modelo_ann_credit_score.keras")')

print(f'\n{"-"*50}')
print(f'  🎉 TALLER COMPLETADO')
print(f'{"-"*50}')
print(f'  Accuracy final  : {acc_test*100:.2f}%')
print(f'  Épocas          : {len(historia.history["loss"])}')
print(f'  Componentes PCA : {n_components}')